# Discrete Attractor Networks: Storing and Recalling Memories

In this exercise, we will build a minimal attractor network model of memory. The central idea is that a memory is not stored as an explicit index, but as a **stable state** of a recurrent neural system. When the network is initialized with a noisy or incomplete cue, it self-organizes the activity pattern back toward a previously stored state. This is the basic mechanism of **pattern completion**, a common model of memory recall in the brain.

We will start with a discrete Hopfield network, where each unit has binary activity $s_i \in \{-1, +1\}$. The network state is the vector

$$
s = (s_1, s_2, \ldots, s_N),
$$

and the recurrent weight matrix $W$ determines how neurons influence each other. If the weights are chosen correctly, stored patterns become attractors of the dynamics: nearby states flow back toward them.

The exercise has four goals:

1. **Implement recall dynamics** in a recurrent binary network.
2. **Understand the Hopfield energy function** as a measure of how well the current state agrees with the pairwise constraints encoded in $W$.
3. **Move from discrete updates to continuous rate dynamics**, where activity relaxes gradually over time.
4. **Implement Hebbian learning** and examine its limitations, including interference, finite memory capacity, and spurious attractors.

The model is deliberately simple. It is not a realistic model of cortex or hippocampus. Instead, it exposes a general computational principle: recurrent networks can store information in the geometry of their state space.

In [ ]:
%pip install numpy matplotlib ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

rng = np.random.default_rng(0)

SHAPE = (10, 10)            # patterns are shown as 10x10 squares
N = SHAPE[0] * SHAPE[1]     # 100 neurons
P = 5                       # stored patterns

SHAPE = (10, 10)            # patterns are shown as 10x10 squares
N = SHAPE[0] * SHAPE[1]     # 100 neurons

_GLYPHS = {
    '0': ['..######..', '.##....##.', '##......##', '##......##', '##......##',
          '##......##', '##......##', '##......##', '.##....##.', '..######..'],
    '1': ['....##....', '...###....', '..####....', '....##....', '....##....',
          '....##....', '....##....', '....##....', '..######..', '.########.'],
    '3': ['########..', '.......##.', '......##..', '...####...', '...####...',
          '......##..', '.......##.', '##.....##.', '.#######..', '..#####...'],
    '4': ['......##..', '.....###..', '....####..', '...##.##..', '..##..##..',
          '.##...##..', '##########', '##########', '......##..', '......##..'],
    '7': ['##########', '##########', '.......##.', '......##..', '.....##...',
          '....##....', '...##.....', '..##......', '.##.......', '##........'],
}
PATTERN_NAMES = ['0', '1', '3', '4', '7']

def _glyph_to_vec(rows):
    return np.array([[1.0 if ch == '#' else -1.0 for ch in r] for r in rows]).flatten()

patterns = np.array([_glyph_to_vec(_GLYPHS[k]) for k in PATTERN_NAMES])
P = len(patterns)
print('patterns:', patterns.shape, ' neurons:', N, ' names:', PATTERN_NAMES)

### Provided helpers
You do not need to modify these functions.

In [ ]:
def overlap(a, b):
    return float(np.dot(a, b) / len(a))

def corrupt(pattern, frac=0.25):
    """Flip a fraction of signs (noise)."""
    s = pattern.copy()
    k = int(frac * len(s))
    idx = rng.choice(len(s), size=k, replace=False); s[idx] *= -1
    return s

def occlude(pattern, frac=0.5):
    """Zero out the bottom `frac` of the image (missing data, not wrong data)."""
    s = pattern.copy()
    cut = int((1 - frac) * len(s))
    s[cut:] = 0.0
    return s

def show_state(s, ax=None, title=None):
    ax = ax or plt.subplots(figsize=(2.2, 2.2))[1]
    ax.imshow(np.reshape(s, SHAPE), cmap='gray', vmin=-1, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=10)
    return ax

def show_patterns(P_arr, titles=None, suptitle=None):
    P_arr = np.atleast_2d(P_arr)
    n = len(P_arr)
    fig, axes = plt.subplots(1, n, figsize=(2*n, 2.2))
    axes = np.atleast_1d(axes)
    for k, ax in enumerate(axes):
        show_state(P_arr[k], ax, titles[k] if titles else None)
    if suptitle: fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

show_patterns(patterns, [f'pattern {i+1}' for i in range(P)],
              suptitle='The 5 memories stored in the network')


## Part 1 — Discrete Hopfield dynamics

In this part of the exercise, we are implementing a simple model of associative memory called a Hopfield network. This kind of network can recreate learned stimuli from partial inputs, making it a useful model for human *content-adressable* memories.

`W` below comes from `hopfield_weights(patterns)`. **Do not look inside it yet.** $W_{ij}$ is the weight from neuron $j$ to neuron $i$.

The energy of a recurrent system measures how stable the current state of the network is. 

- Just like biological neurons sum the inputs from the dendritic compartments in the soma, each simulated neuron sums input $h_i=\sum_j W_{ij}s_j$. For a binary unit, what deterministic rule maps that to $\pm1$ so a stored pattern maps to itself? *Implement `update(state, W)` as a function of the current state and the recurrent connections.*
- Define the energy function. The derivation of this formula is non-trivial, but may be seen intuitively from the fact that we want to measure how much our current state agrees with the future state. In other words we want to compare $s_k$ with $s_{k+1}$. Implement `energy(state, W)` to measure the convergence of the network.

In [ ]:
def hopfield_weights(patterns):
    P, N = patterns.shape
    W = (patterns.T @ patterns) / N
    np.fill_diagonal(W, 0.0)
    return W

W = hopfield_weights(patterns)


In [ ]:
# TODO (you): discrete Hopfield update + energy.
def update(state, W):
    raise NotImplementedError

def energy(state, W):
    raise NotImplementedError

def recall(cue, W, steps=20, record=False):
    """Discrete updates. Stops once a fixed point is reached."""
    s = cue.copy()
    states = [s.copy()]
    energies = [energy(s, W)]
    for _ in range(steps):
        ns = update(s, W)
        states.append(ns.copy())
        energies.append(energy(ns, W))
        if np.array_equal(ns, s):   # fixed point
            break
        s = ns
    return (np.array(states), np.array(energies)) if record else states[-1]


In [ ]:
# Denoising test (run after filling in update/energy)
target = patterns[3]

# Change this to introduce more noise
noise = 0.25

cue = corrupt(target, frac=noise)
states, E = recall(cue, W, steps=20, record=True)
final = states[-1]

show_patterns([target, cue, final], ['target', 'cue ({:.0%} flipped)'.format(noise), 'recalled'])
print('cue overlap:     ', round(overlap(cue, target), 3))
print('recalled overlap:', round(overlap(final, target), 3))
print('steps to settle: ', len(states) - 2)

plt.figure(figsize=(5, 3))
plt.plot(E, marker='o', ms=4)
plt.xlabel('update step')
plt.ylabel(r'$E(s) = -\frac{1}{2} s^\top W s$')
plt.title('Energy of the discrete model')
plt.tight_layout()
plt.show()


### Pattern completion
Suppose that we have an uncorrupted but partial pattern. Can the network complete this too?

In [ ]:
target = patterns[0]
cue = occlude(target, frac=0.79)
states, E = recall(cue, W, steps=20, record=True)
show_patterns([target, cue, states[-1]], ['target', 'stimulus', 'recalled'])
print('completed overlap:', round(overlap(states[-1], target), 3))

target = patterns[2]
cue = occlude(target, frac=0.79)
states, E = recall(cue, W, steps=20, record=True)
show_patterns([target, cue, states[-1]], ['target', 'stimulus', 'recalled'])
print('completed overlap:', round(overlap(states[-1], target), 3))

**Check your understanding (Part 1).**
- Push `corrupt` to 45%. Does it still land on the right pattern, or a spurious/wrong one? Now try 60%.
- Start from a *random* $\pm1$ state — which basin do you fall into, and is $-\xi^\mu$ also stable?
- Why might the negative of a stored pattern also be stable for symmetric binary patterns?

## Part 2 — Continuous-time rate neurons, and the learning rule

Unlike real neurons that have complex dynamics over time, there are no real dynamics in the discrete model. If we want a better model of human memory, we need to account for these factors. The simplest biologically realistic model we can use is a rate neuron. We can define the neuron as a dynamical system where the mean firing rate over some small time $t$ varies according to:

$$\tau\,\dot s_i = -s_i + \phi\!\Big(\beta\sum_j W_{ij}s_j\Big),\qquad \phi=\tanh,\ s_i\in(-1,1).$$

In our case, we are omitting inhibitory interneurons, and so we are allowing the neural rates to be negivate. For a real computational model, this would not be the case.

We integrate it with forward Euler, $s \leftarrow s + \tfrac{\Delta t}{\tau}\big(-s + \phi(\beta W s)\big)$, at small $\Delta t/\tau$. Note that in the limit $\Delta t/\tau\to1$, $\beta\to\infty$, this is the same system as we already implemented in part 1.

In [ ]:
def rate_step(s, W, beta=2.0, dt_tau=0.1):
    """One forward-Euler step of tau*ds/dt = -s + tanh(beta * W s)."""
    return s + dt_tau * (-s + np.tanh(beta * (W @ s)))

def relax(cue, W, beta=2.0, dt_tau=0.1, steps=5000, tol=1e-8, record=False):
    s = cue.copy().astype(float)
    states = [s.copy()]
    energies = [rate_energy(s, W, beta)]
    for _ in range(steps):
        ns = rate_step(s, W, beta, dt_tau)
        states.append(ns.copy())
        energies.append(rate_energy(ns, W, beta))
        if np.max(np.abs(ns - s)) < tol:
            s = ns
            break
        s = ns
    return (np.array(states), np.array(energies)) if record else s

### The correct energy for the rate network

The bare quadratic $-\tfrac12 s^\top W s$ is the Lyapunov function of the *discrete* model — it is **not** guaranteed under rate dynamics. The rate network (Hopfield 1984, *Neurons with graded response have collective computational properties like those of two-state neurons*) has its own Lyapunov function with a leak term:

$$E(s) = -\tfrac12\,s^\top W s + \frac{1}{\beta}\sum_i \int_0^{s_i}\phi^{-1}(u)\,du,\qquad \int_0^{s}\mathrm{arctanh}(u)\,du = s\,\mathrm{arctanh}(s) + \tfrac12\ln(1-s^2).$$

For symmetric $W$ this $E$ is non-increasing along the flow. 

In [ ]:
def rate_energy(s, W, beta=2.0):
    """Graded (Hopfield-1984) Lyapunov function"""
    sc = np.clip(s, -0.999999, 0.999999)
    leak = sc * np.arctanh(sc) + 0.5 * np.log(1 - sc**2)
    return float(-0.5 * s @ W @ s + (1.0 / beta) * np.sum(leak))

### Watch it relax
At $\beta=2$, $\Delta t/\tau=0.1$ the network takes hundreds of small steps to settle. This is a more realistic model of memory in biological circuits, where pattern completion takes time and we can analyze the trajectories in neural state space.

In [ ]:
cue = corrupt(patterns[0], frac=0.30).astype(float)
states, E = relax(cue, W, beta=2.0, dt_tau=0.1, steps=100, record=True)
print('integration steps to settle:', len(states) - 1)
print('mean |s| at fixed point:', round(float(np.mean(np.abs(states[-1]))), 3),
      '  (note: < 1, we are appraoching the discrete attractor)')
print('sign overlap with target:', round(overlap(np.sign(states[-1]), patterns[0]), 3))
print('graded energy monotone non-increasing:', bool(np.all(np.diff(E) <= 1e-7)))

# side-by-side animation: settling state + descending graded energy
sub = max(1, (len(states) - 1) // 80)            # ~80 frames
frames = states[::sub]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(6.6, 3.0), gridspec_kw={'width_ratios':[1,1.5]}, constrained_layout=True)
im = axL.imshow(frames[0].reshape(SHAPE), cmap='gray', vmin=-1, vmax=1)
axL.set_xticks([])
axL.set_yticks([])
axL.set_title('rate relaxation', fontsize=10)
axR.plot(E, color='#E8A87C', lw=1.5)
axR.set_xlabel('Euler step')
axR.set_ylabel('graded energy')
axR.set_title('energy descends', fontsize=10)
cursor = axR.axvline(0, color='#6B82B8', lw=2)

def fr(t):
    im.set_data(frames[t].reshape(SHAPE))
    cursor.set_xdata([min(t*sub, len(E)-1)]*2)
    return [im, cursor]
anim = animation.FuncAnimation(fig, fr, frames=len(frames), interval=80, blit=True)
plt.close(fig); HTML(anim.to_jshtml())

### Gain controls how close the attractors sit to the $\pm1$ corners
Here we explore the range of possible $\beta$ values. Below a critical gain the only fixed point is $s=0$ (the system cannot recall anything if the inputs are too weak). Above it, we can recover an **approximate** representation of the original memory.

In [ ]:
print(f'{"beta":>5} {"mean|s|":>9} {"sign overlap":>13} {"raw overlap":>12}')
betas = np.linspace(0.8, 2.0, 50)
mean = []
sign_overlap = []
raw_overlap = []
for beta in betas:
    cue = corrupt(patterns[0], 0.25).astype(float)
    s = relax(cue, W, beta=beta, dt_tau=0.2, steps=1000)
    mean.append(np.mean(np.abs(s)))
    sign_overlap.append(overlap(np.sign(s), patterns[0]))
    raw_overlap.append(overlap(s, patterns[0]))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(betas, mean, '-', label='mean |s|')
ax.plot(betas, sign_overlap, '-', label='sign overlap')
ax.plot(betas, raw_overlap, '-', label='raw overlap')
ax.set_xlabel('beta')
ax.set_ylabel('mean |s|')
ax.set_title('Rate relaxation')
ax.legend()
plt.show()

### Now the learning rule

In real biological circuits, we do not know what memories to store in the connections *a priori*. Instead, we must learn this from experience. Classical neuroscience can give us a hint here: *neurons that fire together wire together*!

Instead of being handed `W`, the network sees each pattern repeatedly and must *grow* the weights.

A useful learning rule must satisfy three constraints:

1. It is local: $\Delta W_{ij}$ can depend only on the activities of neuron $i$ and neuron $j$.
2. It strengthens connections between units with the same sign.
3. It weakens or makes inhibitory the connections between units with opposite signs.

What simple function of $\xi_i^\mu$ and $\xi_j^\mu$ has this sign structure?

Your learned `W` should reproduce the previsus analytically derived matrix.

In [ ]:
# TODO (you): implement the learning rule
def learn(patterns, epochs=1, lr=1.0/N):
    P, N = patterns.shape
    W = np.zeros((N, N))
    for _ in range(epochs):
        for mu in range(P):
            xi = patterns[mu]
            raise NotImplementedError
    np.fill_diagonal(W, 0)
    return W


In [ ]:
W_learned = learn(patterns)
print('max |W_learned - W_blackbox|:', round(np.max(np.abs(W_learned - W)), 6))

cue = corrupt(patterns[3], 0.30).astype(float)
s = relax(cue, W_learned, beta=2.0, dt_tau=0.1)
show_patterns([patterns[3], cue, np.sign(s)], ['target', 'cue', 'recalled (learned W, rate dyn.)'])
print('sign overlap:', round(overlap(np.sign(s), patterns[3]), 3))

stable = sum(overlap(np.sign(relax(patterns[m].astype(float), W_learned, beta=3.0)), patterns[m]) > 0.99
             for m in range(P))
print(f'patterns recovered: {stable}/{P}')

---
## Part 3 — Exploration (if you finish early)

### 3.1 Memory capacity
The Hopfield network stores $\approx0.138\,N$ patterns before interference destabilizes them (Amit, Gutfreund & Sompolinsky 1985, *Storing infinite numbers of patterns in a spin-glass model of neural networks*). This $\alpha_c$ is a property of the discrete $\pm1$ model; the graded network's capacity is gain-dependent and does not reduce to this clean number, which is why we use the discrete model for the analysis here. With $N=100$, expect breakdown near $P\approx14$.

In [ ]:
def fraction_stable(N, P, trials=15):
    tot = 0
    for _ in range(trials):
        pats = rng.choice([-1.0, 1.0], size=(P, N))
        Wc = learn(pats)
        for m in range(P):
            s = pats[m].copy()
            for _ in range(10):
                s = np.sign(Wc @ s)
            tot += overlap(s, pats[m]) > 0.99
    return tot / (trials * P)

Ps = [5, 8, 10, 12, 14, 16, 20, 26]
frac = [fraction_stable(100, p) for p in Ps]
plt.figure(figsize=(5.5, 3.5))
plt.plot(np.array(Ps)/100, frac, marker='o')
plt.axvline(0.138, ls='--', c='r', label=r'$\alpha_c\approx0.138$')
plt.xlabel(r'load  $\alpha = P/N$')
plt.ylabel('fraction of patterns stable')
plt.title('Capacity of the Hebbian rule (N=100)')
plt.legend()
plt.tight_layout()
plt.show()


### 3.2 Spurious mixture states (discrete model)
Odd mixtures such as $\mathrm{sgn}(\xi^1+\xi^2+\xi^3)$ are also stable fixed points in the network. These can appear as invented memories. If the input stimulus is a combination of already known patterns, then the network may not be able to settle into any memorized state and may become stuck in some mixture of the learned memories.

In [ ]:
mix = np.sign(patterns[0] + patterns[1] + patterns[2])
# Spurious states are fixed points of the SIGN dynamics (the beta->inf limit).
s = mix.copy()
for _ in range(50):
    s = np.sign(W @ s)
print('mixture stable? self-overlap after dynamics:', round(overlap(s, mix), 3))
print('best overlap with any single pattern:',
      round(max(abs(overlap(mix, patterns[k])) for k in range(P)), 3))
show_patterns([patterns[0], patterns[1], patterns[2], mix],
              ['Memory 1', 'Memory 2', 'Memory 3', 'Mixture'])


### 3.3 Strong memories
Presenting a pattern multiple times deepens and widens its basin, so it wins from a larger set of cues. Try storing one pattern 3x alongside the others and check it captures random initial states more often. 

In [ ]:
# Embed pattern 0 three times (a 'strong' memory), then start from random states.
strong_patterns = np.vstack([patterns[0], patterns[0], patterns[0], patterns[1:]])
Ws = learn(strong_patterns)
wins = 0
trials = 40
for _ in range(trials):
    s = rng.choice([-1.0, 1.0], size=N)
    for _ in range(20):
        s = np.sign(Ws @ s)
    if overlap(s, patterns[0]) > 0.99:
        wins += 1
print(f'random init landed on the strong memory {wins}/{trials} times')